In [1]:
import pandas as pd
import numpy as np
import pandera as pa
from pandera import Column, DataFrameSchema, Check
import warnings
warnings.filterwarnings('ignore')

In [2]:
train_df = pd.read_csv('../data/raw/train.csv')
test_df = pd.read_csv('../data/raw/test.csv')

print('Train Shape:', train_df.shape)
print('Test Shape:', test_df.shape)

Train Shape: (100000, 28)
Test Shape: (50000, 27)


In [3]:
print('Columns:', train_df.columns.tolist())
print('\nDtypes:\n', train_df.dtypes)
print('\nNull counts:\n', train_df.isnull().sum())
print('\nTarget distribution:\n', train_df['Credit_Score'].value_counts())

Columns: ['ID', 'Customer_ID', 'Month', 'Name', 'Age', 'SSN', 'Occupation', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Type_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age', 'Payment_of_Min_Amount', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance', 'Credit_Score']

Dtypes:
 ID                           object
Customer_ID                  object
Month                        object
Name                         object
Age                          object
SSN                          object
Occupation                   object
Annual_Income                object
Monthly_Inhand_Salary       float64
Num_Bank_Accounts             int64
Num_Credit_Card               int64
Interest_Rate                 int64
Num_of_Loan                  object
Type

Validation
1. Schema Check (Pandera)
What: Are column names correct? Are dtypes correct?
Why: If a column is missing or dtype is wrong, your entire pipeline breaks downstream. Catch it early
2. Nullable Check
What: Which columns allow nulls? Which don't?
Why: Critical columns like Credit_Score (target) should never be null. If they are, training data is corrupt.
3. Duplicate ID Check
What: Are there repeated rows?
Why: Duplicates cause data leakage — same customer in train and test = inflated model performance
4. Valid Target Classes Check
What: Is Credit_Score only Good/Standard/Poor?
Why: Any unexpected value like None, NaN, or typo like "Goood" will break model training

In [4]:
import pandera as pa
from pandera import Column, DataFrameSchema, Check

schema = DataFrameSchema({
    "ID": Column(str, nullable=False),
    "Customer_ID": Column(str, nullable=False),
    "Month": Column(str, nullable=False),
    "Age": Column(str, nullable=False),
    "Occupation": Column(str, nullable=False),
    "Annual_Income": Column(str, nullable=False),
    "Monthly_Inhand_Salary": Column(float, nullable=True),  # 15k nulls allowed
    "Num_Bank_Accounts": Column(int, nullable=False),
    "Num_Credit_Card": Column(int, nullable=False),
    "Interest_Rate": Column(int, nullable=False),
    "Num_of_Loan": Column(str, nullable=False),
    "Type_of_Loan": Column(str, nullable=True),             # 11k nulls allowed
    "Delay_from_due_date": Column(int, nullable=False),
    "Num_of_Delayed_Payment": Column(str, nullable=True),   # 7k nulls allowed
    "Changed_Credit_Limit": Column(str, nullable=False),
    "Num_Credit_Inquiries": Column(float, nullable=True),   # 1965 nulls allowed
    "Credit_Mix": Column(str, nullable=False),
    "Outstanding_Debt": Column(str, nullable=False),
    "Credit_Utilization_Ratio": Column(float, nullable=False),
    "Credit_History_Age": Column(str, nullable=True),       # 9k nulls allowed
    "Payment_of_Min_Amount": Column(str, nullable=False),
    "Total_EMI_per_month": Column(float, nullable=False),
    "Amount_invested_monthly": Column(str, nullable=True),  # 4479 nulls allowed
    "Payment_Behaviour": Column(str, nullable=False),
    "Monthly_Balance": Column(object, nullable=True),       # 1200 nulls allowed
    "Credit_Score": Column(str, 
        checks=Check.isin(["Good", "Standard", "Poor"]),
        nullable=False
    ),
})

In [5]:
try:
    schema.validate(train_df, lazy=True)
    print("✅ Validation Passed!")
except pa.errors.SchemaErrors as e:
    print("❌ Validation Failed!")
    print(e.failure_cases)

✅ Validation Passed!


In [6]:
def validate_custom_rules(df):
    # Check 1: No duplicate IDs
    dups = df['ID'].duplicated().sum()
    print(f"{'✅' if dups == 0 else '❌'} Duplicate IDs: {dups}")
    
    # Check 2: Valid target classes only
    valid_classes = {"Good", "Standard", "Poor"}
    invalid = set(df['Credit_Score'].unique()) - valid_classes
    print(f"{'✅' if not invalid else '❌'} Invalid target classes: {invalid if invalid else 'None'}")

validate_custom_rules(train_df)

✅ Duplicate IDs: 0
✅ Invalid target classes: None
